In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format='retina'

In [6]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import esm
from tqdm import tqdm

In [7]:
# ============================================================
# Config
# ============================================================
EMB_PATH = "/n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/protein_analysis/ProteoMap/resource/vorf_esm2_embeddings.pt"
VORF_CSV = "/n/holylfs06/LABS/zhuang_lab/Lab/Jiaqi/virome_project/data/Elledge_vORF_pool_complete/AA_to_II_pools.csv"
TORCH_HOME = "/n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/perturbation_analysis/resource/esm"
FASTA_PATH = "/n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/protein_analysis/ProteoMap/resource/vorf_sequences.fasta"

# ============================================================
# 1) Get perturb list
# ============================================================

# option 1: get perturb list from adata
# perturb_list = adata_dom.obs["perturb_id"].unique().tolist()
# perturb_list = [ast.literal_eval(x)[0] if isinstance(x, str) and x.startswith("[") else x for x in perturb_list]
# perturb_list = set(map(str, perturb_list))

# option 2: get perturb list from training dataset directory
# perturb_dir = "/n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/protein_analysis/ProteoMap/data/viral_protein_localization/model_data_real/RPE1/train"
# perturb_list = [d for d in os.listdir(perturb_dir) if os.path.isdir(os.path.join(perturb_dir, d))]

# option 3: get perturb list from the heatmap axis
# perturb_list = list(mat_umap_ordered.columns)

# option 4: get perturb list for full ORFeome
VORF_CSV = "/n/holylfs06/LABS/zhuang_lab/Lab/Jiaqi/virome_project/data/Elledge_vORF_pool_complete/AA_to_II_pools.csv"
vorf_library = pd.read_csv(VORF_CSV)
perturb_list = vorf_library['Fragment ID'].tolist()

# ============================================================
# 2) Load vORF library
# ============================================================
vorf_library = pd.read_csv(VORF_CSV)

# Fill missing "no methionine" sequences using Fragment_sequence[1:]
mask = vorf_library["Fragment_sequence_(no_methionine)"].isna()
vorf_library.loc[mask, "Fragment_sequence_(no_methionine)"] = (vorf_library.loc[mask, "Fragment_sequence"].astype(str).str[1:])

# Filter to perturb list
vorf_library = (vorf_library.loc[vorf_library["Fragment ID"].astype(str).isin(perturb_list),["Fragment ID", "Fragment_sequence_(no_methionine)"],].copy().reset_index(drop=True))
vorf_library["Fragment ID"] = vorf_library["Fragment ID"].astype(str)
vorf_library = vorf_library.dropna(subset=["Fragment_sequence_(no_methionine)"])

assert vorf_library["Fragment ID"].is_unique
assert vorf_library["Fragment_sequence_(no_methionine)"].notna().all()

# Manual include
from Bio.Seq import Seq
manual_sequences = {
    "ffLuc2": "ATGGAAGATGCCAAAAACATTAAGAAGGGCCCAGCGCCATTCTACCCACTCGAAGACGGGACCGCCGGCGAGCAGCTGCACAAAGCCATGAAGCGCTACGCCCTGGTGCCCGGCACCATCGCCTTTACCGACGCACATATCGAGGTGGACATTACCTACGCCGAGTACTTCGAGATGAGCGTTCGGCTGGCAGAAGCTATGAAGCGCTATGGGCTGAATACAAACCATCGGATCGTGGTGTGCAGCGAGAATAGCTTGCAGTTCTTCATGCCCGTGTTGGGTGCCCTGTTCATCGGTGTGGCTGTGGCCCCAGCTAACGACATCTACAACGAGCGCGAGCTGCTGAACAGCATGGGCATCAGCCAGCCCACCGTCGTATTCGTGAGCAAGAAAGGGCTGCAAAAGATCCTCAACGTGCAAAAGAAGCTACCGATCATACAAAAGATCATCATCATGGATAGCAAGACCGACTACCAGGGCTTCCAAAGCATGTACACCTTCGTGACTTCCCATTTGCCACCCGGCTTCAACGAGTACGACTTCGTGCCCGAGAGCTTCGACCGGGACAAAACCATCGCCCTGATCATGAACAGTAGTGGCAGTACCGGATTGCCCAAGGGCGTAGCCCTACCGCACCGCACCGCTTGTGTCCGATTCAGTCATGCCCGCGACCCCATCTTCGGCAACCAGATCATCCCCGACACCGCTATCCTCAGCGTGGTGCCATTTCACCACGGCTTCGGCATGTTCACCACGCTGGGCTACTTGATCTGCGGCTTTCGGGTCGTGCTCATGTACCGCTTCGAGGAGGAGCTATTCTTGCGCAGCTTGCAAGACTATAAGATTCAATCTGCCCTGCTGGTGCCCACACTATTTAGCTTCTTCGCTAAGAGCACTCTCATCGACAAGTACGACCTAAGCAACTTGCACGAGATCGCCAGCGGCGGGGCGCCGCTCAGCAAGGAGGTAGGTGAGGCCGTGGCCAAACGCTTCCACCTACCAGGCATCCGCCAGGGCTACGGCCTGACAGAAACAACCAGCGCCATTCTGATCACCCCCGAAGGGGACGACAAGCCTGGCGCAGTAGGCAAGGTGGTGCCCTTCTTCGAGGCTAAGGTGGTGGACTTGGACACCGGTAAGACACTGGGTGTGAACCAGCGCGGCGAGCTGTGCGTCCGTGGCCCCATGATCATGAGCGGCTACGTTAACAACCCCGAGGCTACAAACGCTCTCATCGACAAGGACGGCTGGCTGCACAGCGGCGACATCGCCTACTGGGACGAGGACGAGCACTTCTTCATCGTGGACCGGCTGAAGAGCCTGATCAAATACAAGGGCTACCAGGTAGCCCCAGCCGAACTGGAGAGCATCCTGCTGCAACACCCCAACATCTTCGACGCCGGGGTCGCCGGCCTGCCCGACGACGATGCCGGCGAGCTGCCCGCCGCAGTCGTCGTGCTGGAACACGGTAAAACCATGACCGAGAAGGAGATCGTGGACTATGTGGCCAGCCAGGTTACAACCGCCAAGAAGCTGCGCGGTGGTGTTGTGTTCGTGGACGAGGTGCCTAAAGGACTGACCGGCAAGTTGGACGCCCGCAAGATCCGCGAGATTCTCATTAAGGCCAAGAAGGGCGGCAAGATCGCCGTGTAA",
    "gfp11-mIFP": "ATGAAAATAAAAACCGGAGCGAGGATTCTCGCATTGTCAGCATTGACAACTATGATGTTCTCAGCCAGTGCCCTCGCCAAGATAGAAGAGGGTAAGCTGGTGATTTGGATTAACGGGGATAAGGGCTATAATGGCCTCGCGGAAGTGGGAAAAAAGTTCGAAAAGGACACGGGTATTAAAGTAACGGTTGAGCACCCCGACAAGCTGGAAGAAAAATTTCCGCAAGTTGCCGCCACCGGTGACGGACCAGACATAATATTCTGGGCGCACGATCGCTTCGGTGGCTACGCTCAATCCGGTCTTTTGGCAGAGATAACCCCCGACAAAGCGTTTCAAGATAAACTCTACCCTTTCACATGGGATGCGGTCCGATATAATGGCAAACTTATAGCCTACCCAATCGCGGTAGAAGCCTTGTCATTGATCTACAACAAGGACCTTTTGCCTAATCCGCCGAAAACGTGGGAGGAGATACCTGCACTTGACAAGGAATTGAAAGCCAAAGGTAAATCAGCCCTTATGTTTAATCTTCAGGAACCGTACTTCACATGGCCACTGATTGCGGCGGATGGGGGTTATGCGTTTAAGTATGAGAACGGGAAGTACGATATAAAAGATGTGGGTGTCGACAACGCAGGCGCAAAAGCTGGTCTGACCTTCCTGGTCGATCTTATCAAAAACAAGCATATGAACGCAGATACCGATTATTCTATTGCTGAAGCCGCATTTAACAAAGGTGAAACCGCCATGACTATTAACGGTCCTTGGGCTTGGTCAAATATTGATACCAGTAAGGTTAATTACGGTGTTACGGTGTTGCCTACATTCAAAGGACAGCCCAGCAAGCCGTTTGTTGGGGTTCTGTCAGCGGGGATTAATGCAGCTTCCCCAAATAAGGAGCTGGCGAAGGAGTTCCTCGAAAATTATCTTCTGACCGATGAAGGTCTCGAAGCTGTCAATAAAGATAAGCCCCTGGGAGCTGTGGCGCTGAAATCCTACGAGGAAGAGCTTGCTAAGGACCCGCGAATTGCGGCGACTATGGAGAATGCCCAGAAGGGCGAGATCATGCCTAACATTCCGCAAATGAGTGCTTTCTGGTATGCAGTACGGACAGCTGTAATAAACGCCGCTTCTGGACGGCAAACCGTCGATGAAGCTCTGAAAGATGCTCAAACGAGAATTACAAAG",
    "MBP_ctrl": "ATGAGCGTACCTCTGACTACCTCAGCATTCGGCCACGCCTTTCTGGCTAACTGTGAACGCGAGCAGATCCACCTGGCGGGCTCCATTCAGCCGCACGGTATCCTGCTGGCTGTGAAAGAGCCGGACAACGTGGTGATCCAGGCTTCTATTAACGCTGCGGAGTTCCTGAACACCAACTCTGTTGTTGGCCGTCCGCTGCGTGACCTGGGCGGCGATCTGCCTTTGCAGATCCTGCCGCACCTGAACGGCCCGCTGCACCTGGCTCCGATGACCCTGCGTTGTACCGTGGGTTCTCCGCCGCGTCGTGTGGACTGTACCATTCATCGTCCGTCTAACGGCGGCCTGATCGTAGAACTGGAACCAGCAACCAAGACCACTAACATTGCGCCGGCTCTGGACGGTGCGTTTCATCGTATCACTTCTTCATCCTCCCTGATGGGCCTGTGTGACGAAACCGCGACTATTATCCGTGAGATTACTGGCTACGACCGTGTGATGGTAGTACGTTTCGATGAAGAGGGTAATGGCGAAATTCTGTCCGAACGTCGTCGTGCGGACCTGGAAGCGTTCCTGGGTAACCGCTACCCGGCGTCTACTATTCCGCAGATCGCTCGTCGCCTGTACGAACATAACCGTGTTCGCCTGCTGGTAGATGTGAACTATACTCCGGTTCCGCTACAGCCGCGCATCAGCCCGCTGAACGGTCGTGATCTGGATATGTCCCTGTCTTGCCTGCGCTCTATGTCCCCGATCCACCAGAAATACATGCAGGACATGGGCGTTGGCGCGACCCTGGTTTGCTCTCTGATGGTGTCTGGTCGTCTGTGGGGTCTGATCGCTTGCCACCACTACGAACCGCGCTTCGTTCCGTTCCACATTCGCGCTGCTGGCGAAGCGCTGGCGGAAACTTGTGCGATCCGCATCGCGACGCTGGAGAGCTTTGCACAGTCTCAGTCCAAACTGTACAAGTAA"
}
for name, nt_seq in manual_sequences.items():
    aa = str(Seq(nt_seq).translate(to_stop=True))
    aa_no_m = aa[1:] if aa.startswith("M") else aa
    manual_sequences[name] = aa_no_m  
    if "*" in aa[:-1]:   # stop codon inside ORF
        print(f"Warning: internal stop codon in {name}")

manual_df = pd.DataFrame({"Fragment ID": list(manual_sequences.keys()), "Fragment_sequence_(no_methionine)": list(manual_sequences.values())})
vorf_library = pd.concat([vorf_library, manual_df], ignore_index=True)
vorf_library = vorf_library.drop_duplicates(subset="Fragment ID", keep="first")

with open(FASTA_PATH, "w") as f:
    for _, row in vorf_library.iterrows():
        frag_id = row["Fragment ID"]
        seq = row["Fragment_sequence_(no_methionine)"]
        f.write(f">{frag_id}\n{seq}\n")

print(f"[INFO] Saved FASTA with {len(vorf_library)} sequences → {FASTA_PATH}")

# Check coverage
library_ids = set(vorf_library["Fragment ID"])
perturb_ids = set(perturb_list)
missing = perturb_ids - library_ids

if len(missing) > 0:
    print("Missing ORFs not covered by vORF library + manual:")
    for x in sorted(missing):
        print(x)
else:
    print("All perturbations have sequences.")
    
# ============================================================
# 3) Load existing embeddings (if any)
# ============================================================
if os.path.exists(EMB_PATH):
    existing_dict = torch.load(EMB_PATH, weights_only=False)
    embedded_ids = set(existing_dict.keys())
    print(f"[INFO] Found existing embeddings: {len(embedded_ids)}")
else:
    existing_dict = {}
    embedded_ids = set()

# ============================================================
# 4) Determine which proteins are missing
# ============================================================
all_ids = set(vorf_library["Fragment ID"])
missing_ids = sorted(all_ids - embedded_ids)

print(f"[INFO] Total proteins: {len(all_ids)}")
print(f"[INFO] Already embedded: {len(embedded_ids)}")
print(f"[INFO] Missing: {len(missing_ids)}")

# ============================================================
# 5) If nothing missing → stop cleanly (NOTEBOOK SAFE)
# ============================================================
if len(missing_ids) == 0:
    print("[INFO] All proteins already embedded. Nothing to do.")
else:
    # ========================================================
    # 6) Prepare missing vORFs
    # ========================================================
    print(f"[INFO] Computing embeddings for {len(missing_ids)} new proteins")

    vorf_missing = (
        vorf_library.loc[vorf_library["Fragment ID"].isin(missing_ids)]
        .reset_index(drop=True)
    )

    # ========================================================
    # 7) Initialize ESM
    # ========================================================
    os.environ["TORCH_HOME"] = TORCH_HOME
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    model = model.to(device)
    model.eval()
    batch_converter = alphabet.get_batch_converter()

    # ========================================================
    # 8) Compute embeddings
    # ========================================================
    BATCH_SIZE = 1  # safe for 650M
    new_embeddings = {}

    with torch.no_grad():
        for i in tqdm(range(0, len(vorf_missing), BATCH_SIZE)):
            batch = vorf_missing.iloc[i : i + BATCH_SIZE]

            data = [
                (row["Fragment ID"], row["Fragment_sequence_(no_methionine)"])
                for _, row in batch.iterrows()
            ]

            labels, strs, tokens = batch_converter(data)
            tokens = tokens.to(device)

            out = model(tokens, repr_layers=[33], return_contacts=False)
            reps = out["representations"][33]

            for j, frag_id in enumerate(labels):
                emb = reps[j, 1 : len(strs[j]) + 1].mean(dim=0)
                new_embeddings[str(frag_id)] = emb.cpu().numpy()

    # ========================================================
    # 9) Merge + save dictionary
    # ========================================================

if os.path.exists(EMB_PATH):
    seq_embedding_dict = torch.load(EMB_PATH, weights_only=False)
else:
    seq_embedding_dict = {}

for k, v in new_embeddings.items():
    seq_embedding_dict[str(k)] = v.astype(np.float32)

torch.save(seq_embedding_dict, EMB_PATH)

print(f"[INFO] Saved {len(seq_embedding_dict)} embeddings to {EMB_PATH}")

[INFO] Saved FASTA with 9085 sequences → /n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/protein_analysis/ProteoMap/resource/vorf_sequences.fasta
All perturbations have sequences.
[INFO] Found existing embeddings: 1527
[INFO] Total proteins: 9085
[INFO] Already embedded: 1527
[INFO] Missing: 7558
[INFO] Computing embeddings for 7558 new proteins


100%|██████████| 7558/7558 [10:46:09<00:00,  5.13s/it]  


[INFO] Saved 9085 embeddings to /n/holylfs05/LABS/zhuang_lab/Lab/Jiaqi/protein_analysis/ProteoMap/resource/vorf_esm2_embeddings.pt
